In [ ]:
# !python3 -m pip install -r requirements.txt

In [1]:
import os
from dotenv import load_dotenv
from typing import List, TypedDict, Annotated
import operator
import mlflow
from langchain_openai import AzureChatOpenAI

# --- Load Environment Variables ---
load_dotenv("../.env")

from utils import get_stock_code, fetch_company_news, analyze_sentiment

# set mlflow tracking url 
mlflow.set_tracking_uri("http://20.75.92.162:5000")

# initialize the model
model_name='gpt4o'
model = AzureChatOpenAI(model=model_name)

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [ ]:
# register the prompt to the prompt store
initial_prompt_template ="""Analyze news for {company_name} ({ticker}) and return JSON with:
            - company_name: string
            - stock_code: string
            - newsdesc: string (news summary)
            - sentiment: string (Positive/Negative/Neutral)
            - people_names: list of strings
            - places_names: list of strings
            - other_companies_referred: list of strings
            - related_industries: list of strings
            - market_implications: string
            - confidence_score: float (0.0-1.0)
            
            News: {news}"""

# Register a new prompt
prompt = mlflow.genai.register_prompt(
    name="arun_prakash-sentiment_analysis-prompt",
    template=initial_prompt_template,
    # Optional: Provide a commit message to describe the changes
    commit_message="Initial commit",
    # Optional: Set tags applies to the prompt (across versions)
    tags={
        "author": "Arun Prakash JK",
        "task": "summarization",
        "language": "en",
        'llm': 'gpt-4o-mini'
    },
)

2025/09/23 17:20:07 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: arun_prakash-sentiment_analysis-prompt, version 2


In [5]:
def main(model, company_name):
    mlflow.set_experiment("arun_prakash_market_sentiment_analyzer")
    with mlflow.start_run(run_name=f"sentiment_{company_name.replace(' ', '_')}"):

        ticker = get_stock_code(company_name)
        if "ticker" in ticker:
            mlflow.log_param("ticker",ticker['ticker'])
            mlflow.set_tag("status", "success")
        else:
            mlflow.set_tag("status", "success")
            mlflow.set_tag("error_message", ticker['error'])
            return None
        
        news = fetch_company_news(ticker['ticker'])
        if "news_summary" in news:
            mlflow.log_param("news_summary",news['news_summary'])
            mlflow.set_tag("status", "success")
        else:
            mlflow.set_tag("status", "success")
            mlflow.set_tag("error_message", ticker['error'])
            return None
        
        result = analyze_sentiment(model, company_name, ticker['ticker'], news['news_summary'])
        if "error" not in ticker:
            mlflow.log_dict(result,f"Sentiment_{ticker['ticker']}.json")
            mlflow.set_tag("status", "success")
            return result
        else:
            mlflow.set_tag("status", "success")
            mlflow.set_tag("error_message", result['error'])
            return None


In [2]:
a = get_stock_code("apple")
b = fetch_company_news(a["ticker"])
c = analyze_sentiment(model, 'apple', a["ticker"], b["news_summary"])

🏃 View run raj_sentiment_analysis at: http://20.75.92.162:5000/#/experiments/0/runs/ba03c687a2c9461d82b811fa28b04adc
🧪 View experiment at: http://20.75.92.162:5000/#/experiments/0


In [3]:
c

{'company_name': 'Apple',
 'stock_code': 'AAPL',
 'newsdesc': 'Apple and US Bank have had their separate settlements with the Consumer Financial Protection Bureau terminated.',
 'sentiment': 'Negative',
 'people_names': [],
 'places_names': [],
 'other_companies_referred': ['US Bank'],
 'related_industries': ['Finance', 'Consumer Protection'],
 'market_implications': 'The termination of settlements could indicate potential regulatory issues for Apple, which may lead to investor caution.',
 'confidence_score': 0.7}